In [31]:
import os
import json
import time
import numpy as np
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt
from multiprocessing import Pool
from utils.runner import run_single_aft


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
import os
import json
import numpy as np
from joblib import Parallel, delayed
from utils.runner import run_single_ranking, run_single_aft

# ==========================================
# 自定义 JSON 和 NumPy 编码
# ==========================================
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        if isinstance(obj, np.generic):
            return obj.item()
        return super(NumpyEncoder, self).default(obj)

# ==========================================
# 参数设定 (Hyperparameters)
# ==========================================
NUM_RUNS = 5
NUM_WORKERS = -1
params = {
    'Experiment': 'AFT Survival',
    'm': 10,
    'n': 100,
    'p_prime': 5,
    'p': 20,
    'pc': 0.3,
    'T': 40,
    'W_inner': 5,
    'rho': 1,
    'ic_type': 'bic',
    'lambda_candidates': np.logspace(-1.8, -1.3, 10).tolist(),
    # 'lambda_globl': np.logspace(-1.7, -1.2, 10).tolist(),
    'lambda_dgd': np.logspace(-2.5, -1.5, 10).tolist(),
    'lambda_d_proxgd': np.logspace(-1.8, -1.3, 10).tolist(),
    'noise_type': 'exp',
    'cens_target': 0.4,
    'dgd_lr': 0.1,
    'd_proxgd_lr': 0.1,
    'noise_scale': 1.0,
    'run_U_ADMM': True,
    'run_Global': True,
    'run_DGD': False,
    'run_D_ProxGD': True
}

# 路径与文件
folder = "aft"
tmp_folder = f"{folder}/tmp_results"  # 临时文件夹
os.makedirs(folder, exist_ok=True)
os.makedirs(tmp_folder, exist_ok=True)
cens_str = str(params.get('cens_target', 0.25)).replace('.', '')
filename = f"{folder}/{params['noise_type']}_m{params['m']}_n{params['n']}_p{params['p']}_pc{str(params['pc']).replace('.', '')}_rho{str(params['rho']).replace('.', '')}_cens{cens_str}.json"

# 选择执行函数
runner_fn = run_single_ranking if "Ranking" in params["Experiment"] else run_single_aft

# ==========================================
# 安全的多进程包装器
# ==========================================
def safe_run_and_save(i, current_params):
    try:
        res = runner_fn(i, current_params)
        tmp_file = os.path.join(tmp_folder, f"run_result_{i}.json")
        with open(tmp_file, 'w', encoding='utf-8') as f:
            # 单次记录也使用自定义编码
            json.dump({'run_id': i, 'result': res}, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)
        return res
    except Exception as e:
        print(f"\n[Error] Worker {i} failed: {str(e)}")
        return {"run_id": i, "status": "failed", "error_message": str(e)}

print(f"Starting Parallel Experiments: {NUM_RUNS} runs...")

if __name__ == '__main__':
    results = Parallel(n_jobs=NUM_WORKERS, verbose=10)(
        delayed(safe_run_and_save)(i, params) for i in range(NUM_RUNS)
    )

    # 收集并保存
    try:
        with open(filename, 'w', encoding='utf-8') as f:
            # 用 cls=NumpyEncoder 替换默认
            json.dump({'parameters': params, 'results': results}, f, ensure_ascii=False, indent=4, cls=NumpyEncoder)
        print(f"\nCompleted! Results saved to {filename}")
    except Exception as e:
        print(f"\n[Error] Final save failed: {e}")
        print(f"所有的运行结果已经安全地保存在了 {tmp_folder} 文件夹下。")


Starting Parallel Experiments: 5 runs...


[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.


In [ ]:
# ==========================================
# 实验完成
# ==========================================
print(f"\n实验完成！数据已保存至 {filename}")
print("请打开 exp2_plot_aft.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。\n")



实验完成！数据已保存至 aft/exp_m10_n100_p20_pc03_rho1_cens04.json
请打开 exp2_plot_aft.ipynb 读取该文件以生成 Markdown 表格和收敛曲线。

